# Step 3: Apply idiom steering to the new figurative benchmark

Goal: test whether the idiom-derived steering direction (built in Step 1, saved
at `results/steering_vector_llama3.2-3b.pkl`) alters figurative-vs-literal rates
on **non-idiomatic** figurative expressions (conventional metaphors, novel
metaphors, and similes) in a systematic way, using the same model, decoding
setup, alpha grid, and judge as Step 1 so the comparison is clean.

This notebook does **not** rebuild the vector; it loads the one Step 1 saved.
Requires `results/steering_vector_llama3.2-3b.pkl` to already exist (from
running Step 1, or pulling it from the repo if already committed).

## Setup: clone repo, install deps, import the shared pipeline

In [ ]:
import os

REPO_DIR = "beyond-idiom-steering"
if not os.path.isdir(REPO_DIR):
    # Running fresh in Colab: clone the repo. If you already have the repo
    # mounted (e.g. via Drive or `%cd`), skip this cell and just make sure
    # your working directory is the repo root.
    !git clone https://github.com/Itamarvs/beyond-idiom-steering.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, ".")


In [ ]:
from src.steering_pipeline import *
import pandas as pd


## Mount Google Drive now

Do this first, not right before the judge step: mounting pops an interactive auth modal, and you don't want the whole sweep sitting blocked on a click hours into an unattended run. Results get written directly to Drive later (see the judge-labeling section) so progress survives a full Colab VM disconnect, not just a process crash.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/beyond-idiom-steering-results"
os.makedirs(DRIVE_DIR, exist_ok=True)
local_path = "results/labeled_figurative_results.csv"
drive_path = os.path.join(DRIVE_DIR, "labeled_figurative_results.csv")
if os.path.isfile(local_path):
    shutil.copy(local_path, drive_path)
    print(f"Copied {local_path} -> {drive_path}")
else:
    print("Nothing labeled yet, starting fresh on Drive.")

## 1. Load the saved idiom steering vector

In [ ]:
vec = load_steering_vector("results/steering_vector_llama3.2-3b.pkl")
v_md, s_md = vec["v_md"], vec["s_md"]
assert vec["model"] == MODEL_NAME, (
    f"Vector was built on {vec['model']}, but this notebook is set up for {MODEL_NAME}. "
    "Load the matching generation model or rebuild the vector."
)
print("loaded vector for", vec["model"], "| source_layer:", vec["source_layer"], "| s_md:", vec["s_md"])


## 2. Load the same base generation model used to build the vector

In [ ]:
model, tokenizer, device = load_generation_model(vec["model"])
print("device:", device, "| model:", vec["model"])


## 3. Load the figurative benchmark

In [ ]:
eval_df = pd.read_csv("data/figurative_eval_benchmark.csv")
print(eval_df.shape)
print(eval_df["category"].value_counts())
print(f"gold rows: {eval_df['gold'].sum()} / {len(eval_df)}")
eval_df.head()


## 4. Sanity check: one item per category, unsteered vs. steered(-4.78)

Same intervention as Step 1 (single-token, prompt-pass-only, injected at layer
`INJECTION_LAYER`, vector built at `SOURCE_LAYER`); only the target benchmark
changes.

In [ ]:
for category in eval_df["category"].unique():
    cat_df = eval_df[eval_df["category"] == category]
    gold_cat_df = cat_df[cat_df["gold"]]
    row = (gold_cat_df.iloc[0] if len(gold_cat_df) else cat_df.iloc[0])
    prefix = row["prefix"]
    print("=" * 80)
    print(f"CATEGORY: {category} | EXPRESSION: {row['expression']} | gold: {row['gold']}")
    unsteered = steered_generate_batch(model, tokenizer, device, prefix, v_md, s_md, alpha_factor=0.0, n_samples=1)[0]
    print(f"UNSTEERED : {prefix}{unsteered}")
    steered_lit = steered_generate_batch(model, tokenizer, device, prefix, v_md, s_md, alpha_factor=-4.78, n_samples=1)[0]
    print(f"STEERED(-4.78, literal push): {prefix}{steered_lit}")
    print()

## 4b. Gold-subset smoke test before the full sweep

Runs the exact same generation + judging pipeline as the full sweep, but
restricted to the 20 `gold` rows (one hand-picked, most rigorously
bidirectionally-validated variant per expression). This is a cheap way to
catch pipeline/prompt issues before committing to the full ~4,500-generation
grid, and doubles as a smaller subset for judge calibration (compare judge
labels on this subset against a manual read, same spirit as the paper's
judge-calibration step).

In [ ]:
gold_df = eval_df[eval_df["gold"]].reset_index(drop=True)
print(f"gold smoke test: {len(gold_df)} rows x {len(ALPHA_FACTOR_GRID)} alphas")

gold_raw = run_generation_eval(
    model, tokenizer, device, gold_df, v_md, s_md,
    out_csv="results/raw_figurative_gold_smoketest.csv",
    key_cols=("expression", "variant_id"),
)
print(f"gold smoke test rows: {len(gold_raw)}")

## 5. Full evaluation sweep (resumable; safe to rerun after a disconnect)

In [ ]:
figurative_results = run_generation_eval(
    model, tokenizer, device, eval_df, v_md, s_md,
    out_csv="results/raw_figurative_results.csv",
    key_cols=("expression", "variant_id"),
)
print(f"total rows: {len(figurative_results)}")

## 6. Free the generation model before loading the judge

In [ ]:
del model
torch.cuda.empty_cache()
print("Generation model freed.")


## 7. LLM-as-judge labeling (Qwen3-14B, 4-bit; same judge as Step 1, for comparability)

See Step 1's judge cell for why: the paper used Gemma-4-31B-it via the
OpenRouter API (doesn't fit free-tier Colab self-hosted); this uses Qwen3-14B
in 4-bit, in the paper's candidate pool but not the winner and unvalidated
here, as a capacity step up from the original Qwen2.5-3B-Instruct judge.


**Speed**: batched (`batch_size=16` below); ~4500 items in this benchmark,
so unbatched (~28s/item observed) would take ~35 hours; batching is what
makes this fit a session. Flushes to disk every `checkpoint_every` batches
and resumes from `labeled_figurative_results.csv` on rerun.

**Resiliency**: that only protects against the *process* crashing; a full
Colab VM disconnect takes local disk with it. Drive was already mounted at the top of this notebook, so that isn't
a blocker here: `run_judge_eval` below writes `labeled_figurative_results.csv` there directly, so
progress survives a disconnect.

In [ ]:
judge_model, judge_tokenizer, device = load_judge_model()


In [ ]:
labeled_df = run_judge_eval(
    judge_model, judge_tokenizer, device,
    in_csv="results/raw_figurative_results.csv",
    out_csv=drive_path,
    key_cols=("expression", "variant_id", "alpha_factor", "sample_i"),
    expr_col="expression",
    batch_size=16,
    checkpoint_every=5,
)
summarize_labels(labeled_df, group_cols=["alpha_factor"])

## 8. Break down by category (Conventional Metaphor / Novel Metaphor / Simile)

In [ ]:
summarize_labels(labeled_df, group_cols=["category", "alpha_factor"])


## 8b. Gold-subset breakdown

The `gold` rows are the most rigorously validated prefixes (one per
expression, both readings confirmed natural). Restricting the summary to
this subset gives a higher-confidence read on the steering effect, less
sensitive to any one borderline prefix.

In [ ]:
gold_labeled_df = labeled_df[labeled_df["gold"]]
print(f"gold rows in labeled results: {len(gold_labeled_df)}")
summarize_labels(gold_labeled_df, group_cols=["alpha_factor"])


## 9. Quick idiom-vs-figurative comparison

Full comparison (plots, deltas, tables for the report) lives in
`analysis/compare_idiom_vs_figurative.py`, which runs locally on the committed
CSVs; no GPU needed. This is just an inline sanity check.

In [ ]:
idiom_labeled = pd.read_csv("results/labeled_idiom_results.csv")
print("IDIOMS:")
print(summarize_labels(idiom_labeled, group_cols=["alpha_factor"]))
print()
print("FIGURATIVE BENCHMARK (all categories pooled):")
print(summarize_labels(labeled_df, group_cols=["alpha_factor"]))


## 10. Commit results (optional; run manually, review `git status` first)

In [ ]:
# !git add results/
# !git commit -m "Step 3: figurative benchmark steering results"
# !git push
